In [1]:
import pandas as pd
from pmdarima import auto_arima
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import itertools
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pmdarima as pm
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint, uniform


X_tr = pd.read_csv("C:/Users/qqhao/Downloads/X_train_Wwou3IE.csv")
y_train = pd.read_csv("C:/Users/qqhao/Downloads/y_train_jJtXgMX.csv")
Y_tt = pd.read_csv("C:/Users/qqhao/Downloads/y_random_pt8afo8.csv")
X_tt = pd.read_csv("C:/Users/qqhao/Downloads/X_test_GgyECq8.csv")
data = X_tr
data1 = X_tt

In [2]:
X_train = X_tr
X_test = X_tt

X_train['DELIVERY_START'] = pd.to_datetime(X_train['DELIVERY_START'],utc=True)
X_test['DELIVERY_START'] = pd.to_datetime(X_test['DELIVERY_START'],utc=True)
X_train.set_index('DELIVERY_START', inplace=True)
X_test.set_index('DELIVERY_START', inplace=True)

# Y

In [3]:
Q1 = y_train['spot_id_delta'].quantile(0.25)
Q3 = y_train['spot_id_delta'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = y_train[(y_train['spot_id_delta'] < lower_bound) | (y_train['spot_id_delta'] > upper_bound)].index

for idx in outliers:
    if idx > 0 and idx < len(y_train) - 1:
        y_train.at[idx, 'spot_id_delta'] = (y_train.at[idx - 1, 'spot_id_delta'] + y_train.at[idx + 1, 'spot_id_delta']) / 2

In [4]:
y_train['DELIVERY_START'] = pd.to_datetime(y_train['DELIVERY_START'],utc=True)
y_train.set_index('DELIVERY_START', inplace=True)

# Load_forecast

In [5]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import KFold,GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

load_forecast_column = 'load_forecast'
predicted_spot_price_column = 'predicted_spot_price'
wind_columns = ['wind_power_forecasts_average', 'wind_power_forecasts_std']
solar_columns = ['solar_power_forecasts_average', 'solar_power_forecasts_std']

def fill_with_past_5_cycles_mean(data, column, cycle_length=168):
    target_data = data[column].isnull()
    target_indices = target_data[target_data == True].index
    
    for idx in target_indices:
        row_num = data.index.get_loc(idx)
        values_to_average = []
        
        for i in range(1, 6):
            prev_idx = row_num - (i * cycle_length)
            if prev_idx >= 0:
                values_to_average.append(data.iloc[prev_idx][column])
        
        if values_to_average:
            mean_value = np.mean(values_to_average)
            data.at[idx, column] = mean_value

fill_with_past_5_cycles_mean(X_train, load_forecast_column)
fill_with_past_5_cycles_mean(X_test, load_forecast_column)

def replace_with_arima(data, column):
    missing_indices = np.where(data[column].isnull())[0]
    available_data = data[column].copy()
    
    model = ARIMA(available_data.dropna(), order=(5, 1, 1))
    model_fit = model.fit()
    
    for idx in missing_indices:
        forecast_value = model_fit.forecast(steps=1)[0]
        data.at[data.index[idx], column] = forecast_value

replace_with_arima(X_train, load_forecast_column)
replace_with_arima(X_test, load_forecast_column)

def fill_with_336_mean(data, column):
    target_data = data[column].isnull()
    target_indices = target_data[target_data == True].index
    
    for idx in target_indices:
        row_num = data.index.get_loc(idx)
        prev_idx = row_num - 336
        next_idx = row_num + 336
        
        if prev_idx >= 0 and next_idx < len(data):
            prev_value = data.iloc[prev_idx][column]
            next_value = data.iloc[next_idx][column]
            mean_value = np.mean([prev_value, next_value])
            data.at[idx, column] = mean_value

for column in wind_columns + solar_columns:
    fill_with_336_mean(X_train, column)
    fill_with_336_mean(X_test, column)

for column in X_train.columns:
    if column not in wind_columns + solar_columns + [predicted_spot_price_column, load_forecast_column]:
        X_train[column].interpolate(method='linear', inplace=True)
        X_test[column].interpolate(method='linear', inplace=True)


C:\Users\qqhao\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\qqhao\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\qqhao\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\qqhao\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(d

# predicted_spot_price

In [6]:
def process_predicted_spot_price(X_train, X_test):
    X_combined = pd.concat([X_train, X_test])
    
    original_indices = X_combined.index
    
    X_features = X_combined.drop(columns=['predicted_spot_price'])
    y_target = X_combined['predicted_spot_price']
    
    X_clean = X_features.loc[~y_target.isnull()]
    y_clean = y_target.dropna()
    
    scaler = StandardScaler()
    X_clean_scaled = scaler.fit_transform(X_clean)
    X_features_scaled = scaler.transform(X_features)

    models = {
        'LinearRegression': LinearRegression(),
        'Lasso': GridSearchCV(Lasso(), param_grid={'alpha': np.logspace(-4, 0, 50)}, cv=3),
        'Ridge': GridSearchCV(Ridge(), param_grid={'alpha': np.logspace(-4, 0, 50)}, cv=3),
        'AdaBoost': AdaBoostRegressor(n_estimators=100)
    }

    kf = KFold(n_splits=7)
    mse_values = {}
    predictions = {}

    for name, model in models.items():
        fold_mse = []
        for train_idx, val_idx in kf.split(X_clean_scaled):
            X_train_fold, X_val_fold = X_clean_scaled[train_idx], X_clean_scaled[val_idx]
            y_train_fold, y_val_fold = y_clean.iloc[train_idx], y_clean.iloc[val_idx]
            
            model.fit(X_train_fold, y_train_fold)
            preds = model.predict(X_val_fold)
            fold_mse.append(mean_squared_error(y_val_fold, preds))
        
        mse_values[name] = np.mean(fold_mse)
        predictions[name] = model.predict(X_features_scaled)

    mse_array = np.array(list(mse_values.values()))
    weights = 1 / mse_array
    weights /= np.sum(weights)

    final_prediction = sum(weights[i] * predictions[name] for i, name in enumerate(predictions))

    final_prediction_series = pd.Series(final_prediction, index=X_combined.index)
    X_combined.loc[X_combined['predicted_spot_price'].isnull(), 'predicted_spot_price'] = final_prediction_series.loc[X_combined['predicted_spot_price'].isnull()]

    if X_combined['predicted_spot_price'].isnull().sum() > 0:
        print("Warning: Some missing values in `predicted_spot_price` were not filled.")
    
    X_filled_tr = X_combined.loc[X_train.index]
    X_filled_tt = X_combined.loc[X_test.index]

    return X_filled_tr, X_filled_tt

X_filled_tr, X_filled_tt = process_predicted_spot_price(X_tr, X_tt)

# Random forest

In [7]:
Y_tr = y_train

In [8]:
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

def add_time_features(data):

    data = data.copy()
    data['year'] = data.index.year
    data['month'] = data.index.month
    data['day'] = data.index.day
    data['day_of_week'] = data.index.dayofweek
    data['hour'] = data.index.hour
    return data

def process_and_predict_with_time_feature(X_train, Y_train, X_test):
    random_seed = np.random.randint(1, 10000)
    print(f"Random seed used: {random_seed}")
    
    X_train = add_time_features(X_train)
    X_test = add_time_features(X_test)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf = RandomForestRegressor()
   
    param_dist = {
        'n_estimators': randint(100, 5000)        # Number of trees in the forest, broader range to capture optimal complexity
        'max_depth': [5], # Depth of each tree, None for fully expanded trees
        'min_samples_split': randint(2, 30),         # Minimum samples required to split a node, lower values to capture fine distinctions
        'min_samples_leaf': randint(1, 20),          # Minimum samples required at each leaf, balances complexity
        'max_features': ['sqrt'],            # Number of features to consider when looking for the best split
        'bootstrap': [True],                  # Whether bootstrap samples are used when building trees
        'max_samples': uniform(0.1, 0.9)             # Fraction of samples used if `bootstrap=True`, varies between 50% and 100%
    }


    rf_random_search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_dist,
        n_iter=500,
        cv=5,
        n_jobs=-1,
        verbose=2,
        random_state=random_seed
    )
    rf_random_search.fit(X_train_scaled, Y_train)

    best_rf_model = rf_random_search.best_estimator_
    print(f"Best hyperparameters: {rf_random_search.best_params_}")

    Y_test_pred = best_rf_model.predict(X_test_scaled)
    Y_test_pred_series = pd.Series(Y_test_pred, index=X_test.index, name="predicted_spot_price")

    return Y_test_pred_series, random_seed

X_filled_tr.index = pd.to_datetime(X_filled_tr.index)
X_filled_tt.index = pd.to_datetime(X_filled_tt.index)

Y_tt_pred, used_random_seed = process_and_predict_with_time_feature(X_filled_tr, Y_tr.values.ravel(), X_filled_tt)


Random seed used: 2733
Fitting 5 folds for each of 500 candidates, totalling 2500 fits
Best hyperparameters: {'bootstrap': True, 'max_depth': 5, 'max_features': 'sqrt', 'max_samples': 0.1140767722332906, 'min_samples_leaf': 5, 'min_samples_split': 8, 'n_estimators': 152}


In [9]:
Y_tt_pred.to_csv('Y_tt_pred_with_time16.csv')

Y_tt 的预测已完成并保存到文件，时间特征已包含。使用的随机种子是: 2733


In [14]:
Y_train = Y_tr.values.ravel()

In [15]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform

X_train_split, X_val_split, Y_train_split, Y_val_split = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)

def add_time_features(data):
    data['hour'] = data.index.hour
    data['dayofweek'] = data.index.dayofweek
    return data

X_train_split = add_time_features(X_train_split)
X_val_split = add_time_features(X_val_split)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_split)
X_val_scaled = scaler.transform(X_val_split)

search_spaces = {
    'n_estimators': Integer(100, 5000), 
    'max_depth': Integer(5, 50),        
    'min_samples_split': Integer(2, 30), 
    'min_samples_leaf': Integer(1, 20),  
    'max_features': ['sqrt'],         
    'bootstrap': [True],        
    'max_samples': Real(0,0.999) 
}


random_seed = np.random.randint(1, 10000)
print(f"Random seed used: {random_seed}")


rf = RandomForestRegressor(random_state=random_seed)


bayes_search = BayesSearchCV(
    estimator=rf,
    search_spaces=search_spaces,
    n_iter=50,  
    cv=5,      
    n_jobs=-1,
    random_state=42,
    verbose=2
)


bayes_search.fit(X_train_scaled, Y_train_split.ravel())

print("Best hyperparameters found by Bayesian Optimization:", bayes_search.best_params_)

best_rf_model = bayes_search.best_estimator_
Y_pred = best_rf_model.predict(X_val_scaled)
mse = mean_squared_error(Y_val_split, Y_pred)
print("Validation MSE with best hyperparameters:", mse)

Random seed used: 3819
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 can

C:\Users\qqhao\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best hyperparameters found by Bayesian Optimization: OrderedDict({'bootstrap': True, 'max_depth': 38, 'max_features

In [18]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_filled_tr)
X_test_scaled = scaler.transform(X_filled_tt)

best_params = {
    'bootstrap': True,
    'max_depth': 38,
    'max_features': 'sqrt',
    'max_samples': 0.9,
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 5000
}


best_rf_model = RandomForestRegressor(**best_params, random_state=3819)
best_rf_model.fit(X_train_scaled, Y_tr.values.ravel()) 


Y_tt_pred = best_rf_model.predict(X_test_scaled)

Y_tt_pred_series = pd.Series(Y_tt_pred, index=X_filled_tt.index, name='predicted_Y')
Y_tt_pred_series.to_csv('Y_tt_predictions17.csv')

print("Predictions saved to 'Y_tt_predictions.csv'.")

Predictions saved to 'Y_tt_predictions.csv'.


In [3]:
import pandas as pd
df = pd.read_csv(r'C:\Users\qqhao\Desktop\俩负才取负2.csv')

def row_vote(row):
    negative_count = (row < 0).sum()
    mean_abs_value = row.abs().mean()
    if negative_count > 7:
        return -mean_abs_value
    else:
        return mean_abs_value

result = df.apply(row_vote, axis=1)

result_df = pd.DataFrame(result, columns=['Vote result'])

result_df.to_csv('y_tt_voted_result21.csv', index=False)